# LAB 1 for CS4243: Textures and Materials
## Notebook 3 — DTD attribute prediction
> **Introduction.** This notebook is part of **LAB 1 for CS4243: Textures and Materials**. It trains and evaluates one-vs-rest attribute predictors on official DTD split 1 and visualises where handcrafted colour, Gabor, and edge statistics align—or fail to align—with human texture words.

## What this notebook tests
We use 13 terms spanning orientation, repetition, surface relief, irregular marks, and complex patterns. The primary folder label and joint annotations form a multi-label target. Training, validation, and test membership comes only from the supplied official split files.

## 1. Load the complete DTD manifest
For a responsive lab walkthrough we take three training and two validation images per selected primary term. Students should scale this up after verifying their implementation.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from texturelab.config import FeatureConfig, GaborConfig
from texturelab.data import DTD_ATTRIBUTES, read_manifest
from texturelab.evaluation import evaluate_attributes
from texturelab.features import extract_local_features, global_pool
from texturelab.models import fit_attribute_classifier
from texturelab.supplied import load_image

attributes = list(DTD_ATTRIBUTES)
dtd = read_manifest(ROOT / "manifests" / "dtd_split1.csv")
size = (64, 64)
train_records = sum(
    (
        [r for r in dtd if r.split == "train" and r.attributes[0] == a]
        for a in attributes
    ),
    [],
)
validation_records = sum(
    (
        [r for r in dtd if r.split == "validation" and r.attributes[0] == a]
        for a in attributes
    ),
    [],
)
print(
    "complete manifest:",
    len(dtd),
    "walkthrough train:",
    len(train_records),
    "validation:",
    len(validation_records),
)

## 2. Visualise selected DTD attributes
One validation image per term illustrates the diversity within human-oriented attribute labels. Joint annotations may include several additional terms beyond the title.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.ravel()
for ax, attribute in zip(axes, attributes):
    record = next(r for r in validation_records if r.attributes[0] == attribute)
    ax.imshow(load_image(record.path, size))
    ax.set_title(attribute + "\n" + ", ".join(record.attributes[1:3]), fontsize=9)
    ax.axis("off")
for ax in axes[len(attributes) :]:
    ax.axis("off")
fig.suptitle("Real DTD split-1 validation examples and joint terms")
fig.tight_layout()

## 3. Extract the shared handcrafted representation
Complete both marked cells in this section. Each image must be resized consistently, converted to an aligned local feature map, and globally pooled using the existing `texturelab` tools. Your final arrays must preserve record order so every descriptor remains aligned with its multi-label target.

In [ ]:
config = FeatureConfig(
    gabor=GaborConfig(
        frequencies=(0.10, 0.22),
        orientations=(0, np.pi / 2),
        kernel_size=11,
        pool_size=7,
    )
)

def descriptor(record):
    image = load_image(record.path, size)
    feature_map, _ = extract_local_features(image, config)
    return global_pool(feature_map)

x_train = np.stack([descriptor(record) for record in train_records])
x_validation = np.stack([descriptor(record) for record in validation_records])
y_train = np.asarray([
    [int(attribute in record.attributes) for attribute in attributes]
    for record in train_records
], dtype=int)
y_validation = np.asarray([
    [int(attribute in record.attributes) for attribute in attributes]
    for record in validation_records
], dtype=int)

print("Descriptor shapes:", x_train.shape, x_validation.shape)
print("Target shapes:", y_train.shape, y_validation.shape)
print("Training positives:", dict(zip(attributes, y_train.sum(axis=0))))
assert len(x_train) == len(y_train)
assert len(x_validation) == len(y_validation)
assert np.isfinite(x_train).all() and np.isfinite(x_validation).all()

### PCA view of descriptor overlap
PCA is a visual diagnostic only—not a classifier or a source of model inputs. Complete the marked cell to compare training descriptors by primary attribute. Overlap suggests that a linear predictor may find those attributes difficult to separate.

In [ ]:
combined_descriptors = np.vstack([x_train, x_validation])
projection = PCA(n_components=2).fit_transform(combined_descriptors)
train_projection = projection[:len(x_train)]
primary = np.asarray([record.attributes[0] for record in train_records])

fig, ax = plt.subplots(figsize=(10, 7))
for attribute in attributes:
    selected = primary == attribute
    ax.scatter(
        train_projection[selected, 0],
        train_projection[selected, 1],
        label=attribute,
        alpha=0.8,
    )
ax.set(title="PCA of real DTD handcrafted descriptors", xlabel="PC1", ylabel="PC2")
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()

## 4. Fit scikit-learn one-vs-rest predictors
Use the supplied model helper rather than implementing logistic regression. `StandardScaler` is already fitted inside its training pipeline, and each estimator learns one attribute independently. Fit only on training data, evaluate unchanged validation probabilities, and save a self-describing Task C bundle for Notebook 4.

In [ ]:
attribute_model = fit_attribute_classifier(
    x_train, y_train, attributes, C=2.0
)
validation_probability = attribute_model.predict_proba(x_validation)
attribute_report = evaluate_attributes(
    y_validation, validation_probability, attributes
)
print(attribute_report)

MODEL_DIR = ROOT / "outputs" / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
task_c_model_path = MODEL_DIR / "task_c_attribute_model.joblib"
joblib.dump(
    {
        "attribute_classifier": attribute_model,
        "feature_config": config,
        "attributes": attributes,
        "image_size": size,
    },
    task_c_model_path,
)
print("Saved frozen Task C model to:", task_c_model_path)

## 5. Visualise per-attribute performance
Complete both marked cells. Average precision assesses ranking quality under imbalance, while thresholded F1 additionally depends on the chosen probability cutoff. Report both rather than interpreting either metric in isolation.

In [ ]:
ap = np.asarray([
    attribute_report["per_attribute"][attribute]["average_precision"]
    for attribute in attributes
])
f1 = np.asarray([
    attribute_report["per_attribute"][attribute]["f1"]
    for attribute in attributes
])

x = np.arange(len(attributes))
width = 0.38
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - width / 2, ap, width, label="average precision")
ax.bar(x + width / 2, f1, width, label="F1 at 0.5")
ax.set_xticks(x, attributes, rotation=55, ha="right")
ax.set_ylim(0, 1)
ax.set_title("DTD split-1 validation performance by attribute")
ax.legend()
fig.tight_layout()

## 6. Qualitative prediction inspection
For one validation image from every selected primary class, view the photograph beside all predicted attribute probabilities. Showing all 13 primary classes avoids drawing conclusions from only a few convenient examples and is essential for explaining why orientation-based attributes succeed while complex visual attributes fail.

In [ ]:
indices = [
    next(
        index
        for index, record in enumerate(validation_records)
        if record.attributes[0] == attribute
    )
    for attribute in attributes
]
fig, axes = plt.subplots(len(indices), 2, figsize=(13, 3.5 * len(indices)))
for row_index, index in enumerate(indices):
    record = validation_records[index]
    axes[row_index, 0].imshow(load_image(record.path, size))
    axes[row_index, 0].set_title(
        f"primary: {record.attributes[0]}\nall labels: " + ", ".join(record.attributes)
    )
    axes[row_index, 0].axis("off")
    order = np.argsort(validation_probability[index])
    axes[row_index, 1].barh(
        np.array(attributes)[order], validation_probability[index, order]
    )
    axes[row_index, 1].set_xlim(0, 1)
    axes[row_index, 1].set_title(
        f"predicted probabilities for {record.attributes[0]} example"
    )
fig.tight_layout()